In [1]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()

DATA_DIR = PROJECT_ROOT / "data"

RAW_AUDIO_DIR = DATA_DIR / "raw_calls"
PROCESSED_AUDIO_DIR = DATA_DIR / "processed_audio"
SPEECH_SEGMENTS_DIR = PROCESSED_AUDIO_DIR / "speech_segments"

TRANSCRIPT_DIR = DATA_DIR / "transcripts"
DIARIZATION_DIR = TRANSCRIPT_DIR / "diarization"

for directory in [
    DATA_DIR,
    RAW_AUDIO_DIR,
    PROCESSED_AUDIO_DIR,
    SPEECH_SEGMENTS_DIR,
    TRANSCRIPT_DIR,
    DIARIZATION_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )

print("Project root      :", PROJECT_ROOT)
print("Raw audio         :", RAW_AUDIO_DIR)
print("Processed audio   :", PROCESSED_AUDIO_DIR)
print("Speech segments   :", SPEECH_SEGMENTS_DIR)
print("Transcripts        :", TRANSCRIPT_DIR)
print("Diarization       :", DIARIZATION_DIR)

Project root      : /teamspace/studios/this_studio
Raw audio         : /teamspace/studios/this_studio/data/raw_calls
Processed audio   : /teamspace/studios/this_studio/data/processed_audio
Speech segments   : /teamspace/studios/this_studio/data/processed_audio/speech_segments
Transcripts        : /teamspace/studios/this_studio/data/transcripts
Diarization       : /teamspace/studios/this_studio/data/transcripts/diarization


In [2]:
import torch

print("=" * 50)
print("RUNTIME")
print("=" * 50)

print("PyTorch version :", torch.__version__)
print("CUDA available  :", torch.cuda.is_available())

if torch.cuda.is_available():

    print(
        "GPU             :",
        torch.cuda.get_device_name(0)
    )

    print(
        "VRAM            :",
        round(
            torch.cuda.get_device_properties(
                0
            ).total_memory / 1024**3,
            2
        ),
        "GB"
    )

device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Selected device :", device)

RUNTIME
PyTorch version : 2.8.0+cu128
CUDA available  : False
Selected device : cpu


In [3]:
!pip install -q \
    faster-whisper \
    silero-vad \
    soundfile \
    librosa \
    ffmpeg-python \
    matplotlib \
    pandas \
    numpy \
    pydantic \
    langchain \
    langchain-google-genai \
    pyannote.audio \
    torchcodec

In [4]:
import json
import re
from pathlib import Path
from typing import Optional, List

from pydantic import BaseModel, Field
from langchain_google_genai import ChatGoogleGenerativeAI

In [5]:
INTENT_DIR = TRANSCRIPT_DIR / "intent"

INTENT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Intent directory:")
print(INTENT_DIR)

Intent directory:
/teamspace/studios/this_studio/data/transcripts/intent


In [6]:
transcript_path = TRANSCRIPT_DIR / "call_00_transcript.json"

if not transcript_path.exists():
    raise FileNotFoundError(
        f"Transcript file not found:\n{transcript_path}"
    )

with open(transcript_path, "r", encoding="utf-8") as f:
    transcript_data = json.load(f)

print("Transcript loaded successfully.")
print("Keys:", transcript_data.keys())

Transcript loaded successfully.
Keys: dict_keys(['call_id', 'audio_file', 'language', 'language_probability', 'segments', 'full_transcript'])


In [7]:
print(json.dumps(
    transcript_data,
    ensure_ascii=False,
    indent=2
))

{
  "call_id": "call_00",
  "audio_file": "call_00_standardized.wav",
  "language": "ar",
  "language_probability": 1,
  "segments": [
    {
      "start": 0.0,
      "end": 4.52,
      "text": "مسائرة خيراً لو سمعت أن كنت وظهر على الشعقة في نيو كايرو",
      "normalized_text": "مسائرة خيرا لو سمعت ان كنت وظهر علي الشعقة في نيو كايرو"
    },
    {
      "start": 4.52,
      "end": 9.6,
      "text": "طررت قود وكنت عايزها تبقى رعيبة من المدارس",
      "normalized_text": "طررت قود وكنت عايزها تبقي رعيبة من المدارس"
    },
    {
      "start": 9.6,
      "end": 12.06,
      "text": "والبجتاتي خمسة مليون",
      "normalized_text": "والبجتاتي خمسة مليون"
    }
  ],
  "full_transcript": "مسائرة خيرا لو سمعت ان كنت وظهر علي الشعقة في نيو كايرو طررت قود وكنت عايزها تبقي رعيبة من المدارس والبجتاتي خمسة مليون"
}


In [8]:
def extract_transcript_text(transcript_data: dict) -> str:
    
    segments = transcript_data.get("segments", [])
    
    texts = []
    
    for segment in segments:
        text = segment.get("text", "").strip()
        
        if text:
            texts.append(text)
    
    return " ".join(texts).strip()

In [9]:
conversation_text = extract_transcript_text(
    transcript_data
)

print("Conversation:")
print(conversation_text)

Conversation:
مسائرة خيراً لو سمعت أن كنت وظهر على الشعقة في نيو كايرو طررت قود وكنت عايزها تبقى رعيبة من المدارس والبجتاتي خمسة مليون


In [10]:
def normalize_arabic_text(text: str) -> str:
    
    text = str(text)
    
    # Remove Tatweel
    text = text.replace("ـ", "")
    
    # Normalize Alef variations
    text = text.replace("أ", "ا")
    text = text.replace("إ", "ا")
    text = text.replace("آ", "ا")
    
    # Normalize Yeh
    text = text.replace("ى", "ي")
    
    # Remove Arabic diacritics
    text = re.sub(
        r"[\u0617-\u061A\u064B-\u0652]",
        "",
        text
    )
    
    # Normalize whitespace
    text = re.sub(
        r"\s+",
        " ",
        text
    )
    
    return text.strip()

In [11]:
normalized_conversation = normalize_arabic_text(
    conversation_text
)

print("Normalized conversation:")
print(normalized_conversation)

Normalized conversation:
مسائرة خيرا لو سمعت ان كنت وظهر علي الشعقة في نيو كايرو طررت قود وكنت عايزها تبقي رعيبة من المدارس والبجتاتي خمسة مليون


In [12]:
class IntentDetection(BaseModel):
    
    intent: str = Field(
        description=(
            "The detected customer intent. "
            "Must be one of: "
            "new_search, modify_search, alternative_search, "
            "positive_feedback, negative_feedback, "
            "property_inquiry, booking_request, "
            "general_question, unknown."
        )
    )
    
    confidence: float = Field(
        description=(
            "Confidence score between 0 and 1."
        )
    )
    
    reason: str = Field(
        description=(
            "Short explanation for why this intent "
            "was selected."
        )
    )

In [ ]:
import os
from getpass import getpass

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

if not GOOGLE_API_KEY:
    GOOGLE_API_KEY = getpass("Enter Google API Key: ")

In [15]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.8-flash",
    temperature=0
)

In [16]:
intent_prompt = """
You are an Intent Detection component inside an AI real-estate
sales assistant.

Your job is to identify the PRIMARY INTENT of the customer
from the conversation.

You MUST choose exactly one of these intents:

1. new_search
   The customer wants to start searching for properties.

2. modify_search
   The customer wants to change an existing search requirement.
   Examples:
   - عايز حاجة اكبر
   - عايز سعر اقل
   - عايز عدد غرف اكتر
   - مش عايز القاهرة الجديدة
   - عايزها تشطيب كامل

3. alternative_search
   The customer wants to see OTHER / DIFFERENT property options.
   Examples:
   - عايز اشوف شقق تانية
   - وريني حاجة تانية
   - عندك بدائل؟
   - ممكن اشوف اختيارات تانية؟

4. positive_feedback
   The customer expresses satisfaction or approval
   about a property.

5. negative_feedback
   The customer expresses dissatisfaction or rejection
   about a property.

6. property_inquiry
   The customer asks for information about a property.
   Examples:
   - كام سعرها؟
   - المساحة كام؟
   - فيها كام اوضة؟

7. booking_request
   The customer wants to book, visit, schedule or reserve
   a property.

8. general_question
   A general question that is related to the service
   but does not fit the previous categories.

9. unknown
   The intent cannot be confidently determined.

IMPORTANT RULES:

- Focus on the customer's PRIMARY intent.
- Do not confuse positive feedback with a request for alternatives.
- If the customer says they like a property BUT requests a change,
  prefer modify_search.
  
Example:
"الشقة حلوة بس عايز حاجة أكبر"
=> modify_search

- If the customer explicitly asks to see different properties
  without specifying a change:
  
"عايز اشوف شقق تانية"
=> alternative_search

- Return exactly one intent.
- Confidence must be between 0 and 1.
- The reason must be short.

Customer conversation:

{conversation}
"""

In [17]:
structured_intent_llm = llm.with_structured_output(
    IntentDetection
)

In [18]:
from langchain_core.prompts import ChatPromptTemplate

intent_chain = (
    ChatPromptTemplate.from_template(intent_prompt)
    | structured_intent_llm
)

In [19]:
intent_result = intent_chain.invoke({
    "conversation": normalized_conversation
})

print("Detected Intent:")
print(intent_result)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Detected Intent:
intent='new_search' confidence=0.95 reason='The customer is providing initial property requirements (location, bedrooms, budget) to initiate a search.'


In [20]:
intent_json = intent_result.model_dump()

print(
    json.dumps(
        intent_json,
        ensure_ascii=False,
        indent=2
    )
)

{
  "intent": "new_search",
  "confidence": 0.95,
  "reason": "The customer is providing initial property requirements (location, bedrooms, budget) to initiate a search."
}


In [21]:
intent_output_path = (
    INTENT_DIR / "call_00_intent.json"
)

with open(
    intent_output_path,
    "w",
    encoding="utf-8"
) as f:
    
    json.dump(
        intent_json,
        f,
        ensure_ascii=False,
        indent=2
    )

print("Intent saved to:")
print(intent_output_path)

Intent saved to:
/teamspace/studios/this_studio/data/transcripts/intent/call_00_intent.json


In [22]:
VALID_INTENTS = {
    "new_search",
    "modify_search",
    "alternative_search",
    "positive_feedback",
    "negative_feedback",
    "property_inquiry",
    "booking_request",
    "general_question",
    "unknown"
}

detected_intent = intent_json["intent"]

if detected_intent not in VALID_INTENTS:
    raise ValueError(
        f"Invalid intent detected: {detected_intent}"
    )

print("Intent validation: PASS")
print("Detected:", detected_intent)

Intent validation: PASS
Detected: new_search


In [23]:
confidence = float(
    intent_json["confidence"]
)

if not 0 <= confidence <= 1:
    raise ValueError(
        f"Invalid confidence score: {confidence}"
    )

print("Confidence validation: PASS")
print("Confidence:", confidence)

Confidence validation: PASS
Confidence: 0.95


In [24]:
def detect_intent(
    conversation: str
) -> dict:
    
    normalized_text = normalize_arabic_text(
        conversation
    )
    
    result = intent_chain.invoke({
        "conversation": normalized_text
    })
    
    result_dict = result.model_dump()
    
    detected_intent = result_dict["intent"]
    
    if detected_intent not in VALID_INTENTS:
        detected_intent = "unknown"
        result_dict["intent"] = "unknown"
    
    confidence = float(
        result_dict["confidence"]
    )
    
    confidence = max(
        0.0,
        min(1.0, confidence)
    )
    
    result_dict["confidence"] = confidence
    
    return result_dict

In [25]:
result = detect_intent(
    conversation_text
)

print(
    json.dumps(
        result,
        ensure_ascii=False,
        indent=2
    )
)

{
  "intent": "new_search",
  "confidence": 0.95,
  "reason": "The customer is initiating a search for an apartment by specifying location, number of rooms, proximity to schools, and budget."
}


In [26]:
test_cases = [
    
    "عايز اشوف شقق تانية",
    
    "حلوة بس عايز حاجة اكبر",
    
    "الشقة دي عجبتني",
    
    "لا مش عاجباني",
    
    "كام سعر الشقة دي؟",
    
    "المساحة كام؟",
    
    "عايز احجز ميعاد عشان اشوف الشقة",
    
    "عايز شقة في القاهرة الجديدة",
    
    "مش عارف اختار"
]

In [27]:
for i, text in enumerate(test_cases, start=1):
    
    result = detect_intent(text)
    
    print("=" * 70)
    print(f"TEST {i}")
    print("Input:")
    print(text)
    
    print("\nOutput:")
    print(
        json.dumps(
            result,
            ensure_ascii=False,
            indent=2
        )
    )

GoogleRateLimitError: Error calling model 'gemini-3.8-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.8-flash\nPlease retry in 1h44m32.314669108s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTie

In [ ]:
critical_test = (
    "الشقة دي حلوة بس عايز حاجة اكبر"
)

critical_result = detect_intent(
    critical_test
)

print(
    json.dumps(
        critical_result,
        ensure_ascii=False,
        indent=2
    )
)

In [ ]:
evaluation_results = []

for text in test_cases:
    
    result = detect_intent(text)
    
    evaluation_results.append({
        "input": text,
        **result
    })

evaluation_path = (
    INTENT_DIR / "intent_test_results.json"
)

with open(
    evaluation_path,
    "w",
    encoding="utf-8"
) as f:
    
    json.dump(
        evaluation_results,
        f,
        ensure_ascii=False,
        indent=2
    )

print("Evaluation saved to:")
print(evaluation_path)